# LOB Engine — Visualization

Drives the C++ matching engine and ITCH parser from Python via the `lobpy`
pybind11 module, and renders three views:

1. **Depth chart** — cumulative bid/ask size (the L2 book shape)
2. **Latency histogram** — per-order submit-path latency with p50/p99/p99.9
3. **Order-flow imbalance** — the OFI signal over a synthetic ITCH replay

### Build the module first
```bash
cmake -B build -DCMAKE_BUILD_TYPE=Release && cmake --build build -j
# put the compiled lobpy*.so on PYTHONPATH (e.g. the build dir), then run Jupyter
```

In [ ]:
import sys, os
# Make the repo's python/ importable (make_charts + built lobpy module).
sys.path.insert(0, os.path.abspath(os.path.join('..', 'python')))
sys.path.insert(0, os.path.abspath('..'))          # build dir on PYTHONPATH also works
import lobpy as L
print('lobpy imported:', [n for n in dir(L) if not n.startswith('_')])

## Raw API sanity check
A resting sell, then a crossing buy — the engine returns the trade and updates the book.

In [ ]:
book = L.OrderBook()
book.submit(L.Order(1, L.Side.Sell, L.OrderType.Limit, 10100, 50))
res = book.submit(L.Order(2, L.Side.Buy, L.OrderType.Limit, 10100, 30))
print('risk:', res.risk)
for t in res.trades:
    print(t)
print('best ask:', book.best_ask(), 'remaining depth:', book.depth_at(L.Side.Sell, 10100))

## Charts
`make_charts` holds the plotting logic (single source of truth shared with the
README generator). Each call writes a PNG into `images/`; we display it inline.

In [ ]:
from IPython.display import Image
import make_charts
make_charts.depth_chart()
Image(filename='../images/depth_chart.png')

In [ ]:
make_charts.latency_hist()
Image(filename='../images/latency_hist.png')

In [ ]:
make_charts.ofi_timeseries()
Image(filename='../images/ofi_timeseries.png')